In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from transformers import AutoImageProcessor, AutoModelForImageClassification
from tqdm.auto import tqdm

import torch.nn.functional as F

#sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA_DIR = Path("geo_dataset")  # change this
TRAIN_DIR = DATA_DIR / "train"
HOLDOUT_DIR = DATA_DIR / "holdout_public"
LABELS_PATH = DATA_DIR / "train_labels.csv"

/home/utn/poli22wo/miniconda3/envs/dl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
OUTPUT_DIR = Path("outputs/xyz")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

best_model_path = OUTPUT_DIR / "best_model.pt"
checkpoint_path = OUTPUT_DIR / "training_checkpoint.pt"
history_path = OUTPUT_DIR / "history.csv"

In [3]:
df = pd.read_csv(LABELS_PATH)

print(df.shape)
display(df.head())

(11758, 5)


,filename,country,iso,lat,lng
0,1fcb4a43864244259b7d8f4a00f1e475.jpg,Turkey,TR,40.112290,38.304629
1,742f45b0211c44ffb19ad84931ea519c.jpg,France,FR,48.094103,-1.994316
2,152a13ef249d4efa95c51ed93f026284.jpg,Turkey,TR,41.324741,27.961821
3,81ce4a88bff14fef8420bca42019b12b.jpg,France,FR,47.585855,-2.971004
4,6fbcfe523e1349759e6060d632d52e54.jpg,United_Kingdom,GB,55.698094,-4.305315


Validation Split

In [4]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["country"],
)

print("Training images:", len(train_df))
print("Validation images:", len(val_df))

Training images: 9406
Validation images: 2352


Model Verification

In [5]:
MODEL_NAME = "apple/mobilevitv2-1.0-imagenet1k-256"

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Device:", device)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 269/269 [00:00<00:00, 53632.54it/s]
[transformers] MobileViTV2ForImageClassification LOAD REPORT from: apple/mobilevitv2-1.0-imagenet1k-256
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([3, 512])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([3])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Device: cuda


In [6]:
total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(f"Parameters: {total_params:,}")

assert total_params <= 5_000_000

Parameters: 4,390,380


Image Processor and Dataset

In [7]:
class GeolocationDataset(Dataset):
    def __init__(
        self,
        dataframe,
        image_dir,
        processor,
        transform=None,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.processor = processor
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = self.image_dir / row["filename"]
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        pixel_values = self.processor(
            images=image,
            return_tensors="pt",
        )["pixel_values"].squeeze(0)

        latitude = np.radians(row["lat"])
        longitude = np.radians(row["lng"])

        x = np.cos(latitude) * np.cos(longitude)
        y = np.cos(latitude) * np.sin(longitude)
        z = np.sin(latitude)

        coordinates = torch.tensor(
            [x, y, z],
            dtype=torch.float32,
        )

        return pixel_values, coordinates

In [8]:
train_dataset = GeolocationDataset(
    train_df,
    TRAIN_DIR,
    processor,
    transform=None,
)

val_dataset = GeolocationDataset(
    val_df,
    TRAIN_DIR,
    processor,
    transform=None,
)

In [9]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

#Loss and Optimizer

In [11]:
def loss_function(predicted_xyz, correct_xyz):
    similarity = (
        predicted_xyz * correct_xyz
    ).sum(dim=1)

    return (1 - similarity).mean()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
)

In [12]:
images, coordinates = next(iter(train_loader))

images = images.to(device)
coordinates = coordinates.to(device)

print("Images:", images.shape)
print("Coordinates:", coordinates.shape)

Images: torch.Size([32, 3, 256, 256])
Coordinates: torch.Size([32, 3])


In [13]:
def haversine_km(lat1, lng1, lat2, lng2):
    radius = 6371.0088

    lat1 = np.radians(lat1)
    lng1 = np.radians(lng1)
    lat2 = np.radians(lat2)
    lng2 = np.radians(lng2)

    difference = (
        np.sin((lat2 - lat1) / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin((lng2 - lng1) / 2) ** 2
    )

    return (
        2
        * radius
        * np.arcsin(
            np.sqrt(np.clip(difference, 0, 1))
        )
    )

In [15]:
def xyz_to_latlon(xyz):
    x = xyz[:, 0]
    y = xyz[:, 1]
    z = xyz[:, 2]

    latitude = np.degrees(
        np.arcsin(np.clip(z, -1, 1))
    )

    longitude = np.degrees(
        np.arctan2(y, x)
    )

    return np.column_stack(
        [latitude, longitude]
    )

In [16]:
start_epoch = 0
END_EPOCH = 40

history = []

best_median = float("inf")
best_epoch = 0

epochs_without_improvement = 0
patience = 4

for epoch in range(start_epoch, END_EPOCH):

    # --------------------
    # Training
    # --------------------
    model.train()
    total_training_loss = 0

    training_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{END_EPOCH} - Training",
    )

    for images, coordinates in training_bar:
        images = images.to(device)
        coordinates = coordinates.to(device)

        optimizer.zero_grad()

        raw_predictions = model(
            pixel_values=images
        ).logits

        predictions = F.normalize(
            raw_predictions,
            p=2,
            dim=1,
        )

        loss = loss_function(
            predictions,
            coordinates,
        )

        loss.backward()
        optimizer.step()

        total_training_loss += loss.item()

        training_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    average_training_loss = (
        total_training_loss / len(train_loader)
    )

    # --------------------
    # Validation
    # --------------------
    model.eval()

    total_validation_loss = 0
    all_predictions = []
    all_coordinates = []

    validation_bar = tqdm(
        val_loader,
        desc=f"Epoch {epoch + 1}/{END_EPOCH} - Validation",
    )

    with torch.no_grad():
        for images, coordinates in validation_bar:
            images = images.to(device)
            coordinates = coordinates.to(device)

            raw_predictions = model(
                pixel_values=images
            ).logits

            predictions = F.normalize(
                raw_predictions,
                p=2,
                dim=1,
            )

            loss = loss_function(
                predictions,
                coordinates,
            )

            total_validation_loss += loss.item()

            all_predictions.append(
                predictions.cpu().numpy()
            )

            all_coordinates.append(
                coordinates.cpu().numpy()
            )

    average_validation_loss = (
        total_validation_loss / len(val_loader)
    )

    # Combine validation batches
    all_predictions = np.concatenate(all_predictions)
    all_coordinates = np.concatenate(all_coordinates)

    predictions_degrees = xyz_to_latlon(
        all_predictions
    )

    coordinates_degrees = xyz_to_latlon(
        all_coordinates
    )

    # Calculate geographic distances
    distances = haversine_km(
        coordinates_degrees[:, 0],
        coordinates_degrees[:, 1],
        predictions_degrees[:, 0],
        predictions_degrees[:, 1],
    )

    mean_distance = np.mean(distances)
    median_distance = np.median(distances)
    within_200 = np.mean(distances < 200)
    within_750 = np.mean(distances < 750)

    # Save this epoch's results
    history.append({
        "epoch": epoch + 1,
        "training_loss": average_training_loss,
        "validation_loss": average_validation_loss,
        "mean_km": mean_distance,
        "median_km": median_distance,
        "within_200": within_200,
        "within_750": within_750,
    })

    # Display this epoch's results
    print(f"\nEpoch {epoch + 1} results")
    print(f"Training loss: {average_training_loss:.4f}")
    print(f"Validation loss: {average_validation_loss:.4f}")
    print(f"Mean distance: {mean_distance:.1f} km")
    print(f"Median distance: {median_distance:.1f} km")
    print(f"Within 200 km: {within_200:.2%}")
    print(f"Within 750 km: {within_750:.2%}")

    if median_distance < best_median:
        best_median = median_distance
        best_epoch = epoch + 1

        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            best_model_path,
        )

        print("Saved new best model.")

    else:
        epochs_without_improvement += 1

        print(
            "Epochs without improvement:",
            epochs_without_improvement,
        )

    if epochs_without_improvement >= patience:
        print("Early stopping.")
        break

Epoch 1/40 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.36it/s]



Epoch 1 results
Training loss: 0.2837
Validation loss: 0.0367
Mean distance: 1532.5 km
Median distance: 1434.1 km
Within 200 km: 1.40%
Within 750 km: 17.69%
Saved new best model.


Epoch 2/40 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.34it/s]



Epoch 2 results
Training loss: 0.0246
Validation loss: 0.0225
Mean distance: 1181.7 km
Median distance: 1087.5 km
Within 200 km: 2.47%
Within 750 km: 28.40%
Saved new best model.


Epoch 3/40 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.64it/s]


...


Epoch 38 results
Training loss: 0.0014
Validation loss: 0.0112
Mean distance: 790.0 km
Median distance: 667.0 km
Within 200 km: 7.57%
Within 750 km: 57.23%
Epochs without improvement: 1


Epoch 39/40 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.66it/s]



Epoch 39 results
Training loss: 0.0015
Validation loss: 0.0109
Mean distance: 780.8 km
Median distance: 656.0 km
Within 200 km: 7.57%
Within 750 km: 57.31%
Saved new best model.


Epoch 40/40 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.60it/s]


Epoch 40 results
Training loss: 0.0014
Validation loss: 0.0110
Mean distance: 780.6 km
Median distance: 659.4 km
Within 200 km: 7.40%
Within 750 km: 57.87%
Epochs without improvement: 1


In [17]:
torch.save(
    {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_median": best_median,
        "history": history,
        "best_epoch": best_epoch,
        "patience": patience,
        "epochs_without_improvement": epochs_without_improvement
    },
    checkpoint_path,
)

In [18]:
history_df = pd.DataFrame(history)
history_df.to_csv(history_path, index=False)